# RAG-based Internship Role Recommendation

## Step 0: Prepare Dynamic Inputs

To make the internship recommendation system dynamic, we'll implement code to:
1.  **Upload a PDF resume** and extract its text content.
2.  **Upload a CSV file** containing the internship knowledge base (`role`, `skills`).

After these steps, the RAG retrieval logic will use these dynamically loaded inputs.

In [2]:
# Install necessary libraries for PDF parsing and data handling
!pip install PyPDF2 pandas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 232.6/232.6 kB 5.1 MB/s eta 0:00:00


### Upload User Resume (PDF)

Please upload your resume PDF file. The system will extract the text content from it. The extracted text will be stored in the `USER_RESUME_TEXT` variable.

In [3]:
from google.colab import files
import PyPDF2
import io

print("Please upload your resume PDF file:")
uploaded_resume = files.upload()

# Assuming only one file is uploaded for the resume
resume_filename = list(uploaded_resume.keys())[0]

# Extract text from PDF
USER_RESUME_TEXT = ""
with io.BytesIO(uploaded_resume[resume_filename]) as pdf_file:
    reader = PyPDF2.PdfReader(pdf_file)
    for page_num in range(len(reader.pages)):
        USER_RESUME_TEXT += reader.pages[page_num].extract_text() + "\n"

print(f"\nSuccessfully extracted text from '{resume_filename}'. First 200 characters:\n{USER_RESUME_TEXT[:200]}...")

Please upload your resume PDF file:


Saving updated_sahithi_resume_compressed (2).pdf to updated_sahithi_resume_compressed (2).pdf

Successfully extracted text from 'updated_sahithi_resume_compressed (2).pdf'. First 200 characters:
SAHITHI  GOGULA  
Hyderabad ⋄ India 
sahithigogula@gmail.com  ⋄ +91-9246936240  ⋄ LinkedIn  ⋄ GitHub  
EDUCATION  
BE in Information  Technology , Vasavi  College  of Engineering  2023–2027  
CGPA:  8...


### Upload Internship Knowledge Base (CSV)

Please upload a CSV file containing your internship knowledge base. It should have at least two columns: `role` and `skills`. The parsed data will be stored in the `INTERNSHIP_KB` variable.

In [4]:
import pandas as pd

print("Please upload your internship knowledge base CSV file (must contain 'role' and 'skills' columns):")
uploaded_kb = files.upload()

# Assuming only one file is uploaded for the knowledge base
kb_filename = list(uploaded_kb.keys())[0]

# Load CSV into DataFrame
kb_df = pd.read_csv(io.BytesIO(uploaded_kb[kb_filename]))

# Convert DataFrame to INTERNSHIP_KB format (list of dictionaries)
# Ensure required columns are present
if 'role' not in kb_df.columns or 'skills' not in kb_df.columns:
    raise ValueError("CSV must contain 'role' and 'skills' columns.")

INTERNSHIP_KB = kb_df[['role', 'skills']].to_dict(orient='records')

print(f"\nSuccessfully loaded knowledge base from '{kb_filename}'. First 2 entries:\n{json.dumps(INTERNSHIP_KB[:2], indent=2)}")

Please upload your internship knowledge base CSV file (must contain 'role' and 'skills' columns):


Saving internship_roles_skills_clean_600.csv to internship_roles_skills_clean_600.csv

Successfully loaded knowledge base from 'internship_roles_skills_clean_600.csv'. First 2 entries:
[
  {
    "role": "AI Research Intern",
    "skills": "python sklearn pandas numpy data cleaning eda statistics probability visualization powerbi tableau feature engineering"
  },
  {
    "role": "AI Research Intern",
    "skills": "python machine learning deep learning pandas numpy scipy statistics sql data visualization feature engineering model evaluation regression classification clustering"
  }
]


In [6]:
import json
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# NOTE: INTERNSHIP_KB and USER_RESUME_TEXT are now loaded dynamically from uploaded files
# in previous cells. We will use those globally defined variables here.

# Ensure INTERNSHIP_KB and USER_RESUME_TEXT are available from previous steps
if 'INTERNSHIP_KB' not in globals() or not INTERNSHIP_KB:
    raise ValueError("INTERNSHIP_KB not found. Please upload the CSV first.")
if 'USER_RESUME_TEXT' not in globals() or not USER_RESUME_TEXT:
    raise ValueError("USER_RESUME_TEXT not found. Please upload the PDF resume first.")

# Step 1: Build semantic representation
# Combine: role + skills for each entry
kb_documents = []
for entry in INTERNSHIP_KB:
    kb_documents.append(f"{entry['role']} {entry['skills']}")

# Add the user resume to the documents for TF-IDF vectorization
all_documents = kb_documents + [USER_RESUME_TEXT]

# Step 2: Compare with resume using semantic similarity (TF-IDF + Cosine Similarity)
vectorizer = TfidfVectorizer()
tfidf_matrix = vectorizer.fit_transform(all_documents)

# The last vector in tfidf_matrix corresponds to the USER_RESUME_TEXT
resume_vector = tfidf_matrix[-1]

# Calculate cosine similarity between the resume and each KB document
# We slice tfidf_matrix to get only the KB documents (excluding the resume itself)
cosine_similarities = cosine_similarity(resume_vector, tfidf_matrix[:-1])

# Step 3: Rank roles by similarity score
# Create a list of (role, score) tuples
role_scores = []
for i, score in enumerate(cosine_similarities[0]):
    role_scores.append({
        "role": INTERNSHIP_KB[i]["role"],
        "score": round(float(score), 2) # Convert to float and round to 2 decimal places
    })

# Sort by score in descending order
role_scores.sort(key=lambda x: x["score"], reverse=True)

# Step 4: Select output (TOP 3 MOST RELEVANT ROLES) ensuring uniqueness
top_3_roles = []
seen_roles = set()
for role_score in role_scores:
    if role_score["role"] not in seen_roles:
        top_3_roles.append(role_score)
        seen_roles.add(role_score["role"])
    if len(top_3_roles) >= 3:
        break

# Output in strict JSON array format
print(json.dumps(top_3_roles, indent=2))

[
  {
    "role": "Full Stack Developer Intern",
    "score": 0.11
  },
  {
    "role": "Backend Developer Intern",
    "score": 0.08
  },
  {
    "role": "Associate Software Engineer Intern",
    "score": 0.07
  }
]
